# Full video — EEVEE GPU render
Stop the previous render cell and keep the current runtime connected to reuse installed Blender and the packed scene. This starts the full video, without another benchmark or preview loop.
Same artist-made mesh, rig, animation, prop contacts, camera coverage, textures, lighting objects, 1080x1920 / 30 fps, narration, music, effects and captions. EEVEE uses rasterization and screen-space ray tracing, so indirect lighting, reflections and subsurface shading can differ from Cycles. Shader initialization can make the first frame slower. No measured T4 speed or quality parity is claimed yet.
Requires an actual NVIDIA graphics context through headless OpenGL/EGL. Software rendering is refused; the runner does not install or replace NVIDIA drivers. A missing graphics context produces a diagnostic report.
Uses 64 temporal samples, full-resolution ray-tracing buffers, ray-trace denoising and jittered soft shadows. Separate scene/settings cache. Finished MP4 previews/downloads automatically.
Snow Rig © Blender Foundation | studio.blender.org — CC BY 4.0.

In [ ]:
#@title Render the prepared video on the Colab GPU
from pathlib import Path
import base64, collections, hashlib, io, json, subprocess, sys, time, zipfile
from IPython.display import Video, display
from google.colab import files
PAYLOAD = '...'
EXPECTED_SHA256 = '46489c89342eafe8edfd8ff101ec87b08c4a26a5cb10ffaaeda6e4004e21afe9'
data = base64.b64decode(PAYLOAD)
if hashlib.sha256(data).hexdigest() != EXPECTED_SHA256:
    raise ValueError("The embedded package is damaged. Reopen the latest notebook.")
destination = Path('/content').resolve()

previous = destination/'video_engine/process-status.json'
if previous.is_file():
    previous_state = json.loads(previous.read_text())
    cmdline = Path('/proc')/str(previous_state.get('pid'))/'cmdline'
    if previous_state.get('state') == 'running' and cmdline.is_file() and b'blender' in cmdline.read_bytes().lower():
        raise RuntimeError('Stop the previous render cell first. Keep the runtime connected; do not restart it.')

with zipfile.ZipFile(io.BytesIO(data)) as archive:
    if any(not (destination/n).resolve().is_relative_to(destination) for n in archive.namelist()):
        raise ValueError("Unsafe package member")
    archive.extractall(destination)
del PAYLOAD, data
root = Path('/content/video_engine')
log = root/'renderer.log'
tail = collections.deque(maxlen=70)
started = time.monotonic()
print("EEVEE full-video render: same packed animation and 1080x1920 / 30 fps; 64 temporal samples, screen-space ray tracing and soft shadows. NVIDIA graphics context is required. Lighting/reflections can differ from Cycles.", flush=True)
with log.open('w', encoding='utf-8', buffering=1) as logfile:
    proc = subprocess.Popen([sys.executable, '-u', str(root/'run_colab_render.py'), '--root', str(root), '--renderer', 'eevee'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        for line in proc.stdout:
            logfile.write(line)
            logfile.flush()
            tail.append(line)
            if line.startswith(('GPU_STAGE', 'BUILD_STAGE', 'FRAME_PROGRESS', 'GPU_VIDEO_READY', 'GPU_ERROR', 'JOB_HEARTBEAT', 'FRAME_STARTED', 'FRAME_TIMING', 'FRAME_RENDER_SECONDS')):
                print(line.rstrip(), flush=True)
    except BaseException:
        import os, signal
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGTERM)
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired: os.killpg(proc.pid, signal.SIGKILL)
        raise
    code = proc.wait()
if code:
    print(''.join(tail))
    report = root/'gpu-error-report.zip'
    with zipfile.ZipFile(report, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in [log, root/'GPU_ERROR.txt', root/'gpu-status.json', root/'blender-native.log', root/'process-status.json', root/'graphics-status.json', root/'graphics-device.json', root/'graphics-preflight.log']:
            if path.exists(): archive.write(path, path.name)
    print("Render failed. Send the downloaded error report back to the chat.")
    files.download(str(report))
    raise RuntimeError(f"GPU render failed with exit code {code}. The complete log is in the error report.")
result = json.loads((root/'gpu-status.json').read_text())
output = Path(result['output'])
print(json.dumps({'status':'complete', 'total_seconds':round(time.monotonic()-started, 1),
                  'renderer':result['renderer'], 'render_and_encode_seconds':result['render_and_encode_seconds']}, indent=2))
display(Video(str(output), embed=True))
files.download(str(output))
print("Done. Send the video or the output above back to the chat.")
